# Discord Persona Chat Interface (Unsloth Optimized)

This notebook runs the fine-tuned Mistral NeMo 12B model. It uses the **Unsloth** backend to double inference speed and minimize VRAM usage, making it perfect for free Colab T4 GPUs.

### Features:
* **Typing Simulation:** Responses stream in with a slight delay to mimic human Discord typing.
* **Benchmarking:** Use `/benchmark` to automatically run a list of standard questions and save the outputs to your Google Drive for comparison.
* **Hot-Swapping:** Use `/switch <gdrive_link> [password]` to dynamically load a different personality without restarting the notebook.
* **Dynamic Sliders:** Use `/temp <value>` or `/pen <value>` to change the hallucination/repetition parameters on the fly.


In [ ]:
# ==========================================
# USER CONFIGURATION
# ==========================================

# 1. Adapter Source
GDRIVE_SHARED_LINK = '' # Paste your final_adapter_encrypted.zip link here
DECRYPTION_KEY = ""

# 2. Persona Setup
CLONE_NAME = "Clone"
SYSTEM_PROMPT = "You are a Discord user. Respond in the exact style, tone, and language of the user you were trained on. Do not act like an AI assistant. Use the same formatting, slang, and response lengths as the original user."


In [ ]:
import os
import time
import datetime
import json

# Global Logging Wrapper
master_log = ""
def log_print(*args, **kwargs):
    global master_log
    output = " ".join(map(str, args))
    print(output, **kwargs)
    master_log += output + "\n"

# Timing Wrapper
section_times = {}
def start_timer(section_name):
    section_times[section_name] = {'start_time': time.time()}
    start_str = datetime.datetime.fromtimestamp(section_times[section_name]['start_time']).strftime('%H:%M:%S')
    log_print(f"\n[INFO] Started '{section_name}' at {start_str}")

def stop_timer(section_name):
    if section_name in section_times and 'start_time' in section_times[section_name]:
        end_t = time.time()
        duration = end_t - section_times[section_name]['start_time']
        end_str = datetime.datetime.fromtimestamp(end_t).strftime('%H:%M:%S')
        log_print(f"[INFO] Finished '{section_name}' at {end_str} (Took {duration:.2f}s)")


In [ ]:
start_timer("Install Dependencies")
# Install Unsloth and utilities
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q gdown pyzipper modelscope
stop_timer("Install Dependencies")

In [ ]:
start_timer("Fetch Adapter")
import gdown
import pyzipper
import shutil

local_zip = '/content/adapter.zip'
local_extract_dir = '/content/local_model'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

if GDRIVE_SHARED_LINK:
    log_print("Downloading adapter from Google Drive...")
    file_id = get_gdrive_id(GDRIVE_SHARED_LINK)
    download_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(download_url, local_zip, quiet=False)

    os.makedirs(local_extract_dir, exist_ok=True)
    log_print("Extracting adapter...")
    try:
        if DECRYPTION_KEY:
            with pyzipper.AESZipFile(local_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                zip_ref.extractall(local_extract_dir)
        else:
            with pyzipper.AESZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(local_extract_dir)
        log_print("Adapter successfully extracted.")
    except Exception as e:
        log_print(f"Extraction failed (Check password): {e}")
else:
    log_print("No link provided. Provide a valid link and restart.")

stop_timer("Fetch Adapter")

In [ ]:
start_timer("Load Model (Unsloth Inference Mode)")
import torch
import gc
import os
from unsloth import FastLanguageModel

# Memory cleanup
for var in ['model', 'tokenizer']:
    if var in globals():
        del globals()[var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

log_print("Loading Model via Unsloth (Highly Optimized Inference)...")
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1' # Safe fallback

# Unsloth automatically handles loading the base model defined in the adapter's config!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_extract_dir, # Path to the extracted adapter
    max_seq_length = 2048, # Giving it plenty of breathing room for inference
    dtype = None, # Auto-detect Float16
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model) # Enables 2x faster inference natively
log_print("Unsloth inference kernels activated.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

stop_timer("Load Model (Unsloth Inference Mode)")

In [ ]:
import warnings
import transformers

# Suppress annoying Hugging Face and PyTorch warnings for a clean chat
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

from google.colab import drive
drive.mount('/content/drive')

BENCHMARK_FILE = '/content/drive/MyDrive/soulclone/custom_benchmarks.txt'

def load_benchmarks():
    if os.path.exists(BENCHMARK_FILE):
        with open(BENCHMARK_FILE, 'r') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
            return lines
    return []

def save_benchmark(prompt):
    os.makedirs(os.path.dirname(BENCHMARK_FILE), exist_ok=True)
    with open(BENCHMARK_FILE, 'a') as f:
        f.write(f"{prompt}\n")

def clear_benchmarks():
    if os.path.exists(BENCHMARK_FILE):
        os.remove(BENCHMARK_FILE)
        return True
    return False

def run_benchmark_suite(model, tokenizer, current_user, temp, pen):
    prompts = load_benchmarks()
    if not prompts:
        print("\n[System: No benchmarks found. Just chat normally, and your prompts will be saved for next time!]")
        return
    
    print(f"\n[System: Running {len(prompts)} stored benchmarks...]")
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = f'/content/drive/MyDrive/soulclone/benchmark_results_{timestamp}.txt'
    
    results = []
    for p in prompts:
        print(f"\n{current_user}: {p}")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"[{current_user}]: {p}"}
        ]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=temp, 
            repetition_penalty=pen, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f"{CLONE_NAME}: {response}")
        
        results.append(f"{current_user}: {p}\n{CLONE_NAME}: {response}\n")
        
    with open(out_path, 'w') as f:
        f.write("\n".join(results))
    print(f"\n[System: Benchmark results saved to Drive: {out_path}]")

# Start Chat
print("\n" + "*"*65)
print("Clone is ready! Commands:")
print(" 'quit'  - Exit the chat")
print(" 'reset' - Clear memory and change user")
print(" '/benchmark' - Run stored prompts and save outputs to Drive")
print(" '/benchmark --reset' - Wipe stored custom benchmarks")
print(" '/temp 0.8' - Change temperature on the fly")
print(" '/pen 1.15' - Change repetition penalty on the fly")
print(" '/switch <link> [password]' - Hot-swap to a new persona adapter")
print("*"*65 + "\n")

conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
current_user = input("Enter your username to begin: ").strip()
print(f"\n[Now chatting as {current_user}]\n")

current_temp = 0.8
current_pen = 1.15

while True:
    user_input = input(f"{current_user}: ")
    
    # Block empty messages from polluting context
    if not user_input.strip():
        print("[System: You cannot send an empty message. Please type something.]")
        continue
        
    if user_input.lower() in ['quit', 'exit', 'stop']:
        print("Ending chat.")
        break
    elif user_input.lower() == 'reset':
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Memory cleared.]")
        current_user = input("Enter new username to begin: ").strip()
        print(f"\n[Now chatting as {current_user}]\n")
        continue
    elif user_input.startswith('/temp '):
        try:
            current_temp = float(user_input.split()[1])
            print(f"[Temperature updated to {current_temp}]")
        except: print("[Invalid temp format. Use: /temp 0.8]")
        continue
    elif user_input.startswith('/pen '):
        try:
            current_pen = float(user_input.split()[1])
            print(f"[Penalty updated to {current_pen}]")
        except: print("[Invalid pen format. Use: /pen 1.15]")
        continue
    elif user_input.startswith('/benchmark'):
        if '--reset' in user_input:
            if clear_benchmarks(): print("[System: Custom benchmarks wiped.]")
            else: print("[System: No benchmarks found to wipe.]")
        else:
            run_benchmark_suite(model, tokenizer, current_user, current_temp, current_pen)
        continue
    elif user_input.startswith('/switch'):
        parts = user_input.split()
        if len(parts) < 2:
            print("[System: Usage: /switch <gdrive_link> [password]]")
            continue
        
        new_link = parts[1]
        new_pwd = parts[2] if len(parts) > 2 else ""
        new_zip = '/content/new_adapter.zip'
        new_extract_dir = f'/content/local_model_{int(time.time())}'
        
        print("[System: Hot-swapping adapter via Unsloth. This will take a moment...]")
        try:
            file_id = get_gdrive_id(new_link)
            gdown.download(f'https://drive.google.com/uc?id={file_id}', new_zip, quiet=True)
            
            os.makedirs(new_extract_dir, exist_ok=True)
            if new_pwd:
                with pyzipper.AESZipFile(new_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                    zip_ref.pwd = new_pwd.encode('utf-8')
                    zip_ref.extractall(new_extract_dir)
            else:
                with pyzipper.AESZipFile(new_zip, 'r') as zip_ref:
                    zip_ref.extractall(new_extract_dir)
            
            # Safely clear the old Unsloth model
            del model
            gc.collect()
            torch.cuda.empty_cache()
            
            # Load new model dynamically 
            from unsloth import FastLanguageModel
            model, _ = FastLanguageModel.from_pretrained(
                model_name = new_extract_dir,
                max_seq_length = 2048,
                dtype = None,
                load_in_4bit = True,
            )
            FastLanguageModel.for_inference(model)
            
            print(f"\n[System: Successfully switched to new persona.]")
            conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        except Exception as e:
            print(f"\n[System: Failed to load new adapter: {e}]")
        continue

    # Save prompt to benchmarks if it's new
    if user_input not in load_benchmarks():
        save_benchmark(user_input)

    # Format and tokenize
    conversation_history.append({"role": "user", "content": f"[{current_user}]: {user_input}"})
    input_text = tokenizer.apply_chat_template(conversation_history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    
    # Generate
    outputs = model.generate(
        **inputs, 
        max_new_tokens=150, 
        temperature=current_temp, 
        repetition_penalty=current_pen, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode and format output
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    conversation_history.append({"role": "assistant", "content": response})
    
    # Sliding window memory (Keep system prompt + last 4 turns)
    if len(conversation_history) > 9:  
        conversation_history = [conversation_history[0]] + conversation_history[-8:]
        
    # Typewriter effect to fix line-break visualization
    print(f"{CLONE_NAME}: ", end="")
    time.sleep(0.5)
    
    lines = response.split('\n')
    for i, line in enumerate(lines):
        print(line)
        if i < len(lines) - 1:
            time.sleep(1.0) # Pause between multi-line responses
    print("\n", end="")